## Concept 9 — Self-Correcting Agent

### What is it?
A `create_agent` wrapped in a **critic-retry loop**. After every answer a separate Critic LLM
evaluates quality. If it fails, the agent retries with the critic's feedback. Automatically.

### Why a Separate Critic?
The agent LLM tends to think its own answers are correct.
A separate Critic call is more objective — it evaluates without bias toward the original answer.

### Pipeline
```
Query
  → create_agent → Answer (Attempt 1)
  → Critic LLM:  PASS → return immediately
               : FAIL 'missing X' → agent retries with feedback (Attempt 2)
  → Critic LLM:  PASS → return
               : FAIL → retry (Attempt 3, max)
  → return last answer
```

### Self-Correcting vs ReAct
```
ReAct:             reasons DURING execution (catches mid-step errors)
Self-Correcting:   validates AFTER execution (catches final answer errors)
Use both together for maximum reliability.
```

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.0)
print(f'LLM ready | Tools: {[t.name for t in TOOLS]}')

### Step 1 — Base Agent (create_agent)
This is the same agent from Notebook 3. The self-correcting loop wraps around it.

In [ ]:
base_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=(
        'You are a helpful assistant. '
        'Use tools for math, documentation, and weather. '
        'Answer all parts of the query completely.'
    ),
)

def run_base_agent(query: str, feedback: str = '') -> str:
    prompt = query
    if feedback:
        prompt = f'{query}\n\n[Your previous answer was rejected. Critic feedback: {feedback}. Try again more carefully.]'

    result = base_agent.invoke({'messages': [{'role': 'user', 'content': prompt}]})

    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for call in msg.tool_calls:
                print(f'  [Agent called] {call["name"]}({call["args"]})')

    return result['messages'][-1].content

# Test
print('Base agent answer:')
print(run_base_agent('What is 144 divided by 12?'))

### Step 2 — Critic LLM
An independent LLM call that evaluates the answer and returns PASS or FAIL.

In [ ]:
critic_chain = ChatPromptTemplate.from_messages([
    ('system',
     'You are a strict quality evaluator.\n'
     'Evaluate whether the agent answer fully and correctly addresses the query.\n\n'
     'Criteria:\n'
     '1. Does it address ALL parts of the query?\n'
     '2. Are numerical results accurate?\n'
     '3. Were appropriate tools used (math → calculate, docs → search_docs, weather → get_weather)?\n'
     '4. Is the answer specific (not vague)?\n\n'
     'Reply format:\n'
     'VERDICT: PASS\n'
     'or\n'
     'VERDICT: FAIL\nREASON: [specific issue]'),
    ('human', 'Query: {query}\n\nAgent Answer: {answer}\n\nEvaluate:'),
]) | llm | StrOutputParser()

def critique(query: str, answer: str) -> tuple:
    evaluation = critic_chain.invoke({'query': query, 'answer': answer})
    passed     = 'VERDICT: PASS' in evaluation
    feedback   = ''
    if not passed:
        for line in evaluation.split('\n'):
            if 'REASON' in line:
                feedback = line.replace('REASON:', '').strip()
                break
        if not feedback:
            feedback = evaluation
    return passed, feedback

# Test critic on a good answer
good_ans = run_base_agent('What is 144 divided by 12?')
passed, feedback = critique('What is 144 divided by 12?', good_ans)
print(f'Answer:   {good_ans}')
print(f'Verdict:  {"PASS" if passed else "FAIL"}')
print(f'Feedback: {feedback}')

### Step 3 — Simulate a Failure
Inject a deliberately wrong answer to see the critic catch it.

In [ ]:
passed, feedback = critique('What is 144 divided by 12?', 'The answer is 15.')
print(f'Bad answer verdict: {"PASS" if passed else "FAIL"}')
print(f'Critic feedback: {feedback}')

### Step 4 — Full Self-Correcting Loop

In [ ]:
MAX_RETRIES = 3

def self_correcting_agent(query: str) -> str:
    feedback = ''

    for attempt in range(1, MAX_RETRIES + 1):
        print(f'\n[Attempt {attempt}]')
        answer = run_base_agent(query, feedback)
        short  = answer[:120] + '...' if len(answer) > 120 else answer
        print(f'  Answer: {short}')

        passed, feedback = critique(query, answer)
        print(f'  Critic: {"PASS" if passed else f"FAIL — {feedback}"}')

        if passed:
            return answer

    print('  Max retries reached.')
    return answer

print('=== Self-Correcting Demo ===')
result = self_correcting_agent('What is the weather in Hyderabad?')
print(f'\nFinal: {result}')

### Step 5 — Multi-Part Query (Critic Verifies All Parts)

In [ ]:
result = self_correcting_agent('Search docs for RAG and also calculate 25 multiplied by 4')
print(f'Final: {result}')

### Test — Standard Queries

In [ ]:
run_queries(self_correcting_agent, label='Concept 9 — Self-Correcting Agent')